# Isla-SNN Training 🧠

Spiking Neural Network language model with Spike Synchrony Attention.

**GPU**: L4 (24GB)+ recommended. T4 works for small config.

In [ ]:
!pip install -q torch transformers datasets wandb gdown

In [ ]:
!git clone https://github.com/Heitorkk2/Isla-SNN.git 2>/dev/null || echo 'Already cloned'
%cd Isla-SNN

In [ ]:
import isla
import torch
print(f"isla v{isla.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Dataset

Download pre-tokenized dataset from Google Drive.

In [ ]:
import os
import gdown
import zipfile
from datasets import load_from_disk

GDRIVE_FILE_ID = "11wg0VXYFJyCqNiL6T9UBytkKTjlQh9Wu"
OUTPUT_ZIP = "/root/isla_dataset.zip"
DATASET_PATH = "/root/isla_dataset"

if not os.path.exists(DATASET_PATH):
    print("Downloading dataset...")
    if not os.path.exists(OUTPUT_ZIP):
        gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", OUTPUT_ZIP, quiet=False)
    print("Extracting...")
    with zipfile.ZipFile(OUTPUT_ZIP, "r") as z:
        z.extractall(DATASET_PATH)
    print("Done.")
else:
    print("Dataset already exists, skipping download.")

# verify
ds = load_from_disk(DATASET_PATH)
seq_len = len(ds[0]["input_ids"]) if "input_ids" in ds.column_names else "?"
print(f"Samples:  {len(ds):,}")
print(f"Columns:  {ds.column_names}")
print(f"Seq len:  {seq_len}")
if isinstance(seq_len, int):
    print(f"Total tok: {len(ds) * seq_len:,}")
del ds

## Configuration

In [ ]:
model_config = isla.ModelConfig(
    hidden_dim=256,
    num_layers=4,
    num_heads=4,
    num_timesteps=4,
    max_seq_len=1024,
    target_spike_rate=0.3,
    spike_reg_lambda=0.01,
    compile=True,
)

train_config = isla.TrainConfig(
    lr=3e-4,
    num_epochs=1,
    batch_size=32,
    gradient_accumulation_steps=4,
    bf16=True,
    gradient_checkpointing=True,
    log_every=50,
    eval_every=500,
    wandb=isla.WandbConfig(
        enabled=True,
        project="Isla-SNN",
        run_name="isla-256d-4L",
    ),
    checkpoint=isla.CheckpointConfig(
        output_dir="./outputs/checkpoints",
        save_every=2000,
        # resume_from="./outputs/checkpoints/latest",  # uncomment to resume
    ),
)

data_config = isla.DataConfig(
    dataset_path=DATASET_PATH,
    tokenizer_name="codelion/gpt-2-70m",
    max_seq_len=1024,
    pack_sequences=False,
)

print("Configs ready.")


## Train

In [ ]:
model, tokenizer = isla.train(model_config, train_config, data_config)

## Generate Text

In [ ]:
import isla
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on: {device}")

model, tokenizer = isla.load_model("./outputs/checkpoints/latest", device=device)
model.eval()

prompt = "Who are you?"
print(f"\nPrompt: {prompt}\n---")

with torch.no_grad():
    for word in isla.generate_stream(
        model, 
        tokenizer, 
        prompt,
        max_new_tokens=50, 
        temperature=0.8,
        top_p=0.9, # top_p not natively in the signature? ah generate_stream handles it in kwargs! wait, does it?
        device=device
    ):
        print(word, end="", flush=True)
print()


## Load from Checkpoint

On a fresh session:

In [ ]:
# model, tokenizer = isla.load_model("./outputs/checkpoints/final", device="cuda")
# print(isla.generate(model, tokenizer, "Hello world", device="cuda"))